In [1]:
import os
import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader

In [2]:
import os 
os.getcwd()

'/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation'

In [3]:
class BirdSoundDataset(Dataset):
    def __init__(self, annotations_file, audio_dir, target_sr=16000, duration=3):
        self.annotations = pd.read_csv(annotations_file)
        self.audio_dir = audio_dir
        self.target_sr = target_sr
        self.num_samples = target_sr * duration

        self.labels = self.annotations["species"].unique()
        self.label_to_index = {label: i for i, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        path = self._get_audio_sample_path(index)
        label = self._get_audio_sample_label(index)

        signal, sr = torchaudio.load(path)

        signal = torch.mean(signal, dim=0, keepdim=True)

        if sr != self.target_sr:
            resampler = torchaudio.transforms.Resample(sr, self.target_sr)
            signal = resampler(signal)

        signal = self._fix_length(signal)

        return signal, label

    def _fix_length(self, signal):
        if signal.shape[1] > self.num_samples:
            start = torch.randint(0, signal.shape[1] - self.num_samples, (1,))
            signal = signal[:, start:start + self.num_samples]
        else:
            pad = self.num_samples - signal.shape[1]
            signal = F.pad(signal, (0, pad))
        return signal

    def _get_audio_sample_path(self, index):
        filename = self.annotations.iloc[index]["filename"]
        return os.path.join(self.audio_dir, filename)

    def _get_audio_sample_label(self, index):
        species = self.annotations.iloc[index]["species"]
        return self.label_to_index[species]

In [4]:
import torch.nn as nn

class AudioClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000,
            n_mels=64
        )

        self.db = torchaudio.transforms.AmplitudeToDB()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):

        x = self.mel(x)       # (B, 1, n_mels, time)
        x = self.db(x)

        x = self.cnn(x)       # (B, 64, 1, 1)
        x = x.view(x.size(0), -1)

        x = self.fc(x)
        return x

In [ ]:
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = BirdSoundDataset(
    "/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation/data/dataset_1/bird_songs_metadata.csv",
    "/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation/data/dataset_1/wavfiles/"
)

dataloader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

model = AudioClassifier(num_classes=len(dataset.labels)).to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(150):
    total_loss = 0
    model.train()

    for signals, labels in dataloader:
        signals = signals.to(device)
        labels = labels.to(device)

        outputs = model(signals)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 35.9549
Epoch 2, Loss: 34.8306
Epoch 3, Loss: 34.2732
Epoch 4, Loss: 32.9026
Epoch 5, Loss: 31.0160
Epoch 6, Loss: 29.1744
Epoch 7, Loss: 26.4160
Epoch 8, Loss: 24.6900
Epoch 9, Loss: 22.8034
Epoch 10, Loss: 21.7084
Epoch 11, Loss: 21.2049
Epoch 12, Loss: 20.6047
Epoch 13, Loss: 19.3838
Epoch 14, Loss: 18.7224
Epoch 15, Loss: 18.0527
Epoch 16, Loss: 18.0279
Epoch 17, Loss: 17.4955
Epoch 18, Loss: 16.8085


In [13]:
class BirdMultiLabelDataset(Dataset):
    def __init__(self, annotations_file, audio_dir, target_sr=16000, duration=3, num_birds=2):
        self.annotations = pd.read_csv(annotations_file)
        self.audio_dir = audio_dir
        self.target_sr = target_sr
        self.num_samples = target_sr * duration
        self.num_birds = num_birds

        self.labels = self.annotations["species"].unique()
        self.label_to_index = {label: i for i, label in enumerate(self.labels)}

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        indices = [random.randint(0, len(self.annotations)-1) for _ in range(self.num_birds)]

        signals = []
        label_vector = torch.zeros(len(self.labels))

        for idx in indices:
            path = self._get_audio_sample_path(idx)
            species = self._get_audio_sample_label(idx)

            signal, sr = torchaudio.load(path)

            signal = torch.mean(signal, dim=0, keepdim=True)

            if sr != self.target_sr:
                resampler = torchaudio.transforms.Resample(sr, self.target_sr)
                signal = resampler(signal)

            signal = self._fix_length(signal)

            signals.append(signal)

            label_vector[species] = 1.0

        mixture = sum(signals)
        mixture = mixture / self.num_birds  # normalize

        return mixture, label_vector

    def _fix_length(self, signal):
        if signal.shape[1] > self.num_samples:
            start = torch.randint(0, signal.shape[1] - self.num_samples, (1,))
            signal = signal[:, start:start + self.num_samples]
        else:
            pad = self.num_samples - signal.shape[1]
            signal = F.pad(signal, (0, pad))
        return signal

    def _get_audio_sample_path(self, index):
        filename = self.annotations.iloc[index]["filename"]
        return os.path.join(self.audio_dir, filename)

    def _get_audio_sample_label(self, index):
        species = self.annotations.iloc[index]["species"]
        return self.label_to_index[species]

In [15]:
dataset = BirdMultiLabelDataset(
    annotations_file="/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation/data/dataset_1/bird_songs_metadata.csv",
    audio_dir="/mnt/WorkSpace/Repos/Simulated-Vibration-Guided-Sound-Seperation/notebooks and experimentation/data/dataset_1/wavfiles/",
    num_birds=2
)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [16]:
class MultiLabelBirdClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000,
            n_mels=64
        )
        self.db = torchaudio.transforms.AmplitudeToDB()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        # x: (B, 1, T)
        x = self.mel(x)
        x = self.db(x)

        x = self.cnn(x)
        x = x.view(x.size(0), -1)

        x = self.fc(x)
        return x

In [17]:
model = MultiLabelBirdClassifier(num_classes=len(dataset.labels)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [19]:
import random

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for signals, labels in dataloader:
        signals = signals.to(device)
        labels = labels.to(device)

        outputs = model(signals)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 110.8952
Epoch 2, Loss: 105.0008
Epoch 3, Loss: 95.5185
Epoch 4, Loss: 89.5706
Epoch 5, Loss: 84.2372


In [20]:
model.eval()

signals, labels = next(iter(dataloader))
signals = signals.to(device)

with torch.no_grad():
    outputs = torch.sigmoid(model(signals))

print("Predictions (first sample):")
print(outputs[0])

print("Ground truth:")
print(labels[0])

Predictions (first sample):
tensor([0.4937, 0.0471, 0.1153, 0.7203, 0.6080], device='cuda:0')
Ground truth:
tensor([1., 0., 0., 0., 1.])
